In [1]:
from datetime import datetime
import pandas as pd
import sys
from pathlib import Path

# Adjust this if your structure differs
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

print("Project root added:", PROJECT_ROOT)


_tdy = datetime.today().strftime("%Y-%m-%d")

out = pd.read_csv(f"../results/{_tdy}/all_matchups_reb_predictions.csv")


Project root added: C:\Users\micha\Coding\python\nba_steam\nba_mike


In [2]:
out.to_csv("./test_reb.csv", index=False)
out

,player,team,opp,is_home,game_date,pred_reb,baseline_reb,delta_reb,p_over_baseline_2,p_over_2
0,Chet Holmgren,OKC,DET,0,2026-02-20,9.091206,0.0,9.091206,0.994126,0.994126
1,Tobias Harris,DET,OKC,1,2026-02-20,5.723706,0.0,5.723706,0.958377,0.958377
2,Ausar Thompson,DET,OKC,1,2026-02-20,5.626206,0.0,5.626206,0.955819,0.955819
3,Jaylin Williams,OKC,DET,0,2026-02-20,5.528982,0.0,5.528982,0.953104,0.953104
4,Cade Cunningham,DET,OKC,1,2026-02-20,5.491482,0.0,5.491482,0.952011,0.952011
...,...,...,...,...,...,...,...,...,...,...
85,Cameron Johnson,DEN,BOS,1,2026-02-20,3.606973,0.0,3.606973,0.843930,0.843930
86,Sam Hauser,BOS,DEN,0,2026-02-20,3.593706,0.0,3.593706,0.842616,0.842616
87,Bruce Brown,DEN,BOS,1,2026-02-20,3.361206,0.0,3.361206,0.817703,0.817703
88,Payton Pritchard,BOS,DEN,0,2026-02-20,3.113706,0.0,3.113706,0.786896,0.786896


In [3]:
# ============================
# TEST: Rebounds selectors using test_reb.csv
# ============================

import pandas as pd

from model_training.rebounds.selector import (
    select_jackpot_reb_ticket,
    select_matchup_coverage_reb_ticket,
    assign_pencil_decision_reb,
)

# --- Load uploaded test file ---
df = out.copy()  # Replace with pd.read_csv("test_reb.csv") if loading from file
print(f"Loaded {len(df):,} rows")
print("Columns:", sorted(df.columns.tolist()))

# --- Basic required columns sanity ---
required = {"player","team","opp","is_home","pred_reb","baseline_reb","delta_reb"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required prediction columns: {sorted(missing)}")

# --- Detect baseline prob column ---
p_cols = [c for c in df.columns if c.startswith("p_over_baseline_")]
if not p_cols:
    raise ValueError("No p_over_baseline_* column found.")
p_col = sorted(p_cols)[-1]
print("Using probability column:", p_col)


# ============================================================
# 1) JACKPOT
# ============================================================
jackpot = select_jackpot_reb_ticket(
    df,
    n_legs=3,
    over_baseline_col=p_col,
    min_pred_reb=7.0,
    min_p_over_baseline=0.18,
    min_delta_reb=2.0,
    max_per_team=1,
)

print("\n=== JACKPOT ===")
print(jackpot[["player","team","pred_reb","baseline_reb","delta_reb",p_col]].to_string(index=False))


# ============================================================
# 2) COVERAGE
# ============================================================
coverage = select_matchup_coverage_reb_ticket(
    df,
    players_per_matchup=2,
    insurance_per_matchup=1,
    min_pred_reb=6.0,
    p_col=p_col,
    min_prob=0.18,
    min_delta_reb=0.0,
    max_legs=18,
)

print("\n=== COVERAGE ===")
print(coverage[["player","team","pred_reb","delta_reb",p_col]].to_string(index=False))


# ============================================================
# 3) OVER LINE (if sportsbook column exists)
# ============================================================
if "p_over_9_5" in df.columns:
    over_line = select_over_line_reb_ticket(
        df,
        n_legs=10,
        p_col="p_over_9_5",
        min_pred_reb=6.0,
        min_prob=0.58,
        max_per_team=3,
    )

    print("\n=== OVER LINE ===")
    print(over_line[["player","team","pred_reb","delta_reb","p_over_9_5"]].to_string(index=False))
else:
    print("\n[INFO] No sportsbook probability column (p_over_9_5) found — skipping over-line test.")


# ============================================================
# 4) PENCIL LABELS
# ============================================================


Loaded 90 rows
Columns: ['baseline_reb', 'delta_reb', 'game_date', 'is_home', 'opp', 'p_over_2', 'p_over_baseline_2', 'player', 'pred_reb', 'team']
Using probability column: p_over_baseline_2

=== JACKPOT ===
           player team  pred_reb  baseline_reb  delta_reb  p_over_baseline_2
     Nikola Jokić  DEN 11.506206           0.0  11.506206           0.998365
Victor Wembanyama  SAS  9.938706           0.0   9.938706           0.996293
    Jarrett Allen  CLE  9.443706           0.0   9.443706           0.995157

=== COVERAGE ===
            player team  pred_reb  delta_reb  p_over_baseline_2
      Nikola Jokić  DEN 11.506206  11.506206           0.998365
     Neemias Queta  BOS  8.686206   8.686206           0.992647
    Nikola Vučević  BOS  7.486206   7.486206           0.985442
     Jarrett Allen  CLE  9.443706   9.443706           0.995157
      Jericho Sims  MIL  7.476973   7.476973           0.985363
      Myles Turner  MIL  6.681973   6.681973           0.976647
     Chet Holmgre

In [4]:
df_labeled = assign_pencil_decision_reb(df)

display("\n=== PENCIL COUNTS ===")
display(df_labeled["pencil"].value_counts())

display("\n=== TOP 15 (by pred_reb) ===")
display(
    df_labeled.sort_values("pred_reb", ascending=False)
    [["player","team","pred_reb","delta_reb",p_col,"pencil"]]
    .head(15)
    
)


'\n=== PENCIL COUNTS ==='

pencil
coverage_only    79
over              6
smash             5
Name: count, dtype: int64

'\n=== TOP 15 (by pred_reb) ==='

,player,team,pred_reb,delta_reb,p_over_baseline_2,pencil
76,Nikola Jokić,DEN,11.506206,11.506206,0.998365,smash
15,Victor Wembanyama,SAS,9.938706,9.938706,0.996293,smash
62,Jarrett Allen,CLE,9.443706,9.443706,0.995157,smash
0,Chet Holmgren,OKC,9.091206,9.091206,0.994126,smash
49,Dylan Cardwell(TW),SAC,9.046206,9.046206,0.993979,smash
77,Neemias Queta,BOS,8.686206,8.686206,0.992647,over
50,Alperen Şengün,HOU,8.556973,8.556973,0.992096,over
16,Scottie Barnes,TOR,7.763706,7.763706,0.987598,over
78,Nikola Vučević,BOS,7.486206,7.486206,0.985442,over
63,Jericho Sims,MIL,7.476973,7.476973,0.985363,over
